# FastSpeech 2: English (Svarah) & Telugu (IndicTTS) End-to-End Master Notebook
**IIIT Hyderabad Voice AI Research Internship**

This master notebook provides an end-to-end, GPU-accelerated pipeline on Google Colab:
1. GPU Hardware Audit & Google Drive Mounting (Persistent Storage)
2. Environment Setup & Dependency Installation
3. Codebase Verification (Automated Unit Tests)
4. Dataset Acquisition: Svarah (English) & IndicTTS (Telugu) 5h Train / 1h Val Splits
5. IndicMFA Alignment & Acoustic Feature Extraction
6. Benchmark Training (100–500 steps) & Performance Profiling
7. Full Model Training with Periodic Google Drive Checkpointing
8. Waveform Synthesis on Mandatory Test Sentences (E1–E4, T1–T4)
9. Visualization: 30 Variance Adaptor Plots & 4 Loss Curves

In [ ]:
# 1. GPU Verification and Hardware Audit
!nvidia-smi

import torch
print("PyTorch Version:", torch.__version__)
assert torch.cuda.is_available(), "ERROR: GPU accelerator not detected! Go to Runtime -> Change runtime type -> Select T4 or A100 GPU."
device_name = torch.cuda.get_device_name(0)
total_memory = torch.cuda.get_device_properties(0).total_memory / 1e9
print(f"Active GPU: {device_name} ({total_memory:.2f} GB VRAM)")

In [ ]:
# 2. Mount Google Drive for Persistent Storage of Checkpoints & Audio
from google.colab import drive
import os

drive.mount("/content/drive")
drive_save_dir = "/content/drive/MyDrive/fastspeech2_tts"
os.makedirs(drive_save_dir, exist_ok=True)
print(f"Google Drive persistent directory ready: {drive_save_dir}")

In [ ]:
# 3. Clone Repository & Install Dependencies
!git clone https://github.com/Dhanush1205/fastspeech2-english-telugu-tts.git
%cd fastspeech2-english-telugu-tts

!pip install -q soundfile librosa pyworld tgt g2p-en transformers datasets pyyaml tqdm matplotlib scipy
print("Dependencies installed successfully!")

In [ ]:
# 4. Run Automated Unit & Integration Tests
!python -m unittest tests/test_model.py

In [ ]:
# 5. Dataset Acquisition: Download Svarah & IndicTTS Telugu (5.0h Train / 1.0h Val)
# Svarah Indian English (~5h train / ~1h val)
!python data_prep/prepare_svarah.py \
    --output_dir data/svarah \
    --splits_dir English/data_splits \
    --train_hours 5.0 \
    --val_hours 1.0

# IndicTTS Telugu (~5h train / ~1h val)
!python data_prep/prepare_indictts.py \
    --output_dir data/indictts_telugu \
    --splits_dir Telugu/data_splits \
    --train_hours 5.0 \
    --val_hours 1.0


In [ ]:
# 6. Download AI4Bharat IndicMFA Telugu Acoustic Model & Dictionary
!python Telugu/preprocess.py --download_indicmfa_models

In [ ]:
# 7. Preprocessing & Feature Extraction (Mel, F0, Energy, Duration)
!python English/preprocess.py \
    --config English/config.yaml \
    --data_dir data/svarah \
    --output_dir data/preprocessed/english \
    --train_hours 5.0 \
    --val_hours 1.0

!python Telugu/preprocess.py \
    --config Telugu/config.yaml \
    --data_dir data/indictts_telugu \
    --output_dir data/preprocessed/telugu \
    --train_hours 5.0 \
    --val_hours 1.0

In [ ]:
# 8. Benchmark Training (200 Steps) & Performance Profiling
print("=== Running English Benchmark (200 steps) ===")
!python English/train.py \
    -p English/config/preprocess.yaml \
    -m English/config/model.yaml \
    -t English/config/train.yaml \
    --total_step 200

print("\n=== Running Telugu Benchmark (200 steps) ===")
!python Telugu/train.py \
    -p Telugu/config/preprocess.yaml \
    -m Telugu/config/model.yaml \
    -t Telugu/config/train.yaml \
    --total_step 200


In [ ]:
# 9. Full Training Loop with Google Drive Checkpoint Mirroring
# Run after reviewing benchmark results
!python English/train.py \
    -p English/config/preprocess.yaml \
    -m English/config/model.yaml \
    -t English/config/train.yaml

!python Telugu/train.py \
    -p Telugu/config/preprocess.yaml \
    -m Telugu/config/model.yaml \
    -t Telugu/config/train.yaml

# Sync trained checkpoints to Google Drive
!cp -r English/checkpoints/* /content/drive/MyDrive/fastspeech2_tts/english_checkpoints/
!cp -r Telugu/checkpoints/* /content/drive/MyDrive/fastspeech2_tts/telugu_checkpoints/
print("Checkpoints successfully mirrored to Google Drive!")

In [ ]:
# 10. Synthesize Mandatory Test Sentences (E1-E4, T1-T4)
!python English/inference.py --output_dir Results/English/test_generated_audios
!python Telugu/inference.py --output_dir Results/Telugu/test_generated_audios

# Sync generated audios to Google Drive
!cp -r Results/English/test_generated_audios/* /content/drive/MyDrive/fastspeech2_tts/english_audio/
!cp -r Results/Telugu/test_generated_audios/* /content/drive/MyDrive/fastspeech2_tts/telugu_audio/


In [ ]:
# 11. Generate 30 Variance Adaptor Plots & 4 Loss Curves
!python utils/plot_variance.py
!python utils/plot_losses.py

# Recompile Technical Report with Genuine Experimental Results
!python Report/generate_report.py

In [ ]:
# 12. Audio Playback in Notebook
import IPython.display as ipd
print("English E1 Audio:")
ipd.display(ipd.Audio("Results/English/test_generated_audios/E1.wav"))
print("Telugu T1 Audio:")
ipd.display(ipd.Audio("Results/Telugu/test_generated_audios/T1.wav"))